In [ ]:
import pandas as pd
import numpy as np

# 데이터 불러오기
df = pd.read_csv('./data/model_df_할인율NaN&음수처리.csv')

# 2023년과 2024년 데이터 분리
df_2023 = df[df['기획년도'] == 2023].reset_index(drop=True)
df_2024 = df[df['기획년도'] == 2024].reset_index(drop=True)

# 2023년과 2024년에 존재하는 카테고리 확인
cat_2023 = set(df_2023['카테고리'].unique())
cat_2024 = set(df_2024['카테고리'].unique())

# 1) 2023, 2024 둘 다 존재하는 카테고리
common_cats = cat_2023.intersection(cat_2024)

# 2) 2024년에만 존재하는 카테고리
new_cats_2024 = cat_2024 - cat_2023

# 3) 데이터프레임 분리
df_2024_common = df_2024[df_2024['카테고리'].isin(common_cats)].reset_index(drop=True)
df_2024_new = df_2024[df_2024['카테고리'].isin(new_cats_2024)].reset_index(drop=True)

print(f"2024년 카테고리 수: {len(cat_2024)}")
print(f"2023년과 2024년 공통 카테고리 수: {len(common_cats)}")
print(f"2024년에만 존재하는 신규 카테고리 수: {len(new_cats_2024)}")


2024년 카테고리 수: 91
2023년과 2024년 공통 카테고리 수: 66
2024년에만 존재하는 신규 카테고리 수: 25


In [ ]:
# 시즌별 목표 설정
season_target = {'봄': 0.80, '여름': 0.80, '가을': 0.65, '겨울': 0.55, '사계절': 0.80}

# 주차 정렬
df_2024_common = df_2024_common.sort_values(by=['시즌', '카테고리', '주차']).reset_index(drop=True)
df_2023 = df_2023.sort_values(by=['시즌', '카테고리', '주차']).reset_index(drop=True)

# 공통 카테고리 주차별 목표 누적 판매율 생성
df_2024_common['주차별_목표누적판매율'] = np.nan

for season in df_2024_common['시즌'].unique():
    for category in df_2024_common[df_2024_common['시즌'] == season]['카테고리'].unique():
        # 2023년 동일 시즌·카테고리 패턴 추출
        cat_curve_2023 = df_2023[
            (df_2023['시즌'] == season) & (df_2023['카테고리'] == category)
        ].sort_values(by='주차')['누적판매율'].values

        # 2024년 대상 데이터 추출
        target_data = df_2024_common[
            (df_2024_common['시즌'] == season) & (df_2024_common['카테고리'] == category)
        ]
        total_weeks_2024 = target_data.shape[0]
        goal = season_target[season]

        if len(cat_curve_2023) == 0 or total_weeks_2024 == 0:
            continue  # 혹시라도 빠지는 케이스 방지

        # 2023 패턴 선형 보간
        scaled_curve = np.interp(
            np.linspace(0, len(cat_curve_2023) - 1, total_weeks_2024),
            np.arange(len(cat_curve_2023)),
            cat_curve_2023
        )
        scaled_curve = scaled_curve / scaled_curve[-1] * goal  # 목표값 맞추기

        # log 가중치 생성
        log_curve = np.log(np.linspace(1, total_weeks_2024, total_weeks_2024))
        log_curve = (log_curve - log_curve.min()) / (log_curve.max() - log_curve.min())

        # log 가중치 적용 후 목표 도달 재조정
        log_adjusted_curve = scaled_curve * log_curve
        tail_curve = log_adjusted_curve[1:]
        tail_curve = tail_curve / tail_curve[-1] * goal
        final_curve = np.concatenate([[scaled_curve[0]], tail_curve])

        # 결과 입력
        df_2024_common.loc[
            (df_2024_common['시즌'] == season) & (df_2024_common['카테고리'] == category),
            '주차별_목표누적판매율'
        ] = np.round(final_curve, 4)


In [24]:
# 월별 목표 누적 판매율 컬럼 생성
df_2024_common['월별_목표누적판매율'] = np.nan

for season in df_2024_common['시즌'].unique():
    for category in df_2024_common[df_2024_common['시즌'] == season]['카테고리'].unique():
        for month in df_2024_common[(df_2024_common['시즌'] == season) & (df_2024_common['카테고리'] == category)]['월'].unique():
            # 해당 시즌-카테고리-월 데이터 추출
            month_data = df_2024_common[
                (df_2024_common['시즌'] == season) & 
                (df_2024_common['카테고리'] == category) & 
                (df_2024_common['월'] == month)
            ].sort_values(by='주차')

            if month_data.empty:
                continue

            # 월의 마지막 주차 목표 누적 판매율 추출
            last_week_goal = month_data.iloc[-1]['주차별_목표누적판매율']

            # 월 전체에 동일 목표 적용
            df_2024_common.loc[
                (df_2024_common['시즌'] == season) & 
                (df_2024_common['카테고리'] == category) & 
                (df_2024_common['월'] == month),
                '월별_목표누적판매율'
            ] = last_week_goal

df_2024_common_cat = set(df_2024_common['카테고리'].unique())
len(df_2024_common_cat) # 66개 확인함

66

In [ ]:
# 신규 카테고리도 주차별 정렬
df_2024_new = df_2024_new.sort_values(by=['시즌', '카테고리', '주차']).reset_index(drop=True)
df_2024_new['주차별_목표누적판매율'] = np.nan

for season in df_2024_new['시즌'].unique():
    # 2023년 해당 시즌 전체 평균 패턴 생성
    season_df = df_2023[
        (df_2023['시즌'] == season) & (df_2023['총입고수량'] > 0)
    ].sort_values(by='주차').copy()

    # 누적판매율 평균(주마다 평균)으로 23년도 판매 패턴 뽑기
    season_pattern = season_df.groupby('주차')['누적판매율'].mean().reset_index()
    season_pattern = season_pattern.sort_values(by='주차')['누적판매율'].values

    for category in df_2024_new[df_2024_new['시즌'] == season]['카테고리'].unique():
        target_data = df_2024_new[
            (df_2024_new['시즌'] == season) & (df_2024_new['카테고리'] == category)
        ]
        total_weeks_2024 = target_data.shape[0]
        goal = season_target[season]

        if len(season_pattern) == 0 or total_weeks_2024 == 0:
            continue

        # 시즌 평균 패턴 보간
        scaled_curve = np.interp(
            np.linspace(0, len(season_pattern) - 1, total_weeks_2024),
            np.arange(len(season_pattern)),
            season_pattern
        )
        scaled_curve = scaled_curve / scaled_curve[-1] * goal

        # log 가중치 생성
        log_curve = np.log(np.linspace(1, total_weeks_2024, total_weeks_2024))
        log_curve = (log_curve - log_curve.min()) / (log_curve.max() - log_curve.min())

        # log 가중치 적용 후 목표 맞추기
        log_adjusted_curve = scaled_curve * log_curve # 적용
        tail_curve = log_adjusted_curve[1:]
        tail_curve = tail_curve / tail_curve[-1] * goal
        final_curve = np.concatenate([[scaled_curve[0]], tail_curve])

        # 결과
        df_2024_new.loc[
            (df_2024_new['시즌'] == season) & (df_2024_new['카테고리'] == category),
            '주차별_목표누적판매율'
        ] = np.round(final_curve, 4)

df_2024_new.head(50) #최소 1개의 제품 시즌 마감까지 목표누적판매율 확인

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,...,맑음,흐림,비,강한 비,눈,강한 눈,진눈깨비,악천후일수,평균기온(도),주차별_목표누적판매율
0,2024,2024-06-09,가을_니트 셔츠_티에리_ZB,ZB,01_시즌,가을,니트 셔츠,티에리,1:남성,8767,...,6,0,1,0,0,0,0,0,25.0,0.0000
1,2024,2024-06-16,가을_니트 셔츠_티에리_ZB,ZB,01_시즌,가을,니트 셔츠,티에리,1:남성,8767,...,6,0,1,0,0,0,0,0,26.5,0.0010
2,2024,2024-06-23,가을_니트 셔츠_티에리_ZB,ZB,01_시즌,가을,니트 셔츠,티에리,1:남성,8767,...,4,1,1,1,0,0,0,1,25.3,0.0035
3,2024,2024-06-30,가을_니트 셔츠_티에리_ZB,ZB,01_시즌,가을,니트 셔츠,티에리,1:남성,8767,...,1,0,4,2,0,0,0,2,24.7,0.0075
4,2024,2024-07-07,가을_니트 셔츠_티에리_ZB,ZB,01_시즌,가을,니트 셔츠,티에리,1:남성,8767,...,2,2,2,1,0,0,0,1,25.5,0.0131
5,2024,2024-07-14,가을_니트 셔츠_티에리_ZB,ZB,01_시즌,가을,니트 셔츠,티에리,1:남성,8767,...,0,3,2,2,0,0,0,2,26.8,0.0207
6,2024,2024-07-21,가을_니트 셔츠_티에리_ZB,ZB,01_시즌,가을,니트 셔츠,티에리,1:남성,8767,...,0,1,3,3,0,0,0,3,27.7,0.0320
7,2024,2024-07-28,가을_니트 셔츠_티에리_ZB,ZB,01_시즌,가을,니트 셔츠,티에리,1:남성,8767,...,4,3,0,0,0,0,0,0,29.2,0.0524
8,2024,2024-08-04,가을_니트 셔츠_티에리_ZB,ZB,01_시즌,가을,니트 셔츠,티에리,1:남성,8767,...,3,3,1,0,0,0,0,0,29.8,0.0730
9,2024,2024-08-11,가을_니트 셔츠_티에리_ZB,ZB,01_시즌,가을,니트 셔츠,티에리,1:남성,8767,...,2,2,3,0,0,0,0,0,30.3,0.0998


In [25]:
# 신규 카테고리 월별 목표 누적 판매율 계산
df_2024_new['월별_목표누적판매율'] = np.nan

for season in df_2024_new['시즌'].unique():
    for category in df_2024_new[df_2024_new['시즌'] == season]['카테고리'].unique():
        for month in df_2024_new[(df_2024_new['시즌'] == season) & (df_2024_new['카테고리'] == category)]['월'].unique():
            month_data = df_2024_new[
                (df_2024_new['시즌'] == season) & 
                (df_2024_new['카테고리'] == category) & 
                (df_2024_new['월'] == month)
            ].sort_values(by='주차')

            if month_data.empty:
                continue

            last_week_goal = month_data.iloc[-1]['주차별_목표누적판매율']

            df_2024_new.loc[
                (df_2024_new['시즌'] == season) & 
                (df_2024_new['카테고리'] == category) & 
                (df_2024_new['월'] == month),
                '월별_목표누적판매율'
            ] = last_week_goal

df_2024_new_cat = set(df_2024_new['카테고리'].unique())
len(df_2024_new_cat) # 25개 확인함

25

In [ ]:
# 2023년 데이터는 주차별/월별 목표 컬럼 NaN으로 생성
df_2023['주차별_목표누적판매율'] = np.nan
df_2023['월별_목표누적판매율'] = np.nan

# 공통 카테고리와 신규 카테고리 2024 데이터 합치기
df_2024_final = pd.concat([df_2024_common, df_2024_new], ignore_index=True)

# 각 카테고리별 첫 번째 주차 목표 누적판매율을 0으로 변경
first_week_idx = df_2024_final.groupby(['시즌', '카테고리'])['주차'].idxmin()
df_2024_final.loc[first_week_idx, '주차별_목표누적판매율'] = 0

# 주차별과 월별 목표 누적판매율 컬럼 소수점 2자리로 반올림
df_2024_final['주차별_목표누적판매율'] = (df_2024_final['주차별_목표누적판매율'] * 100).round(2)
df_2024_final['월별_목표누적판매율'] = (df_2024_final['월별_목표누적판매율'] * 100).round(2)

# 2023년과 2024년 최종 합치기
df_2324 = pd.concat([df_2023, df_2024_final], ignore_index=True)

# 데이터 정렬 (최종 확인용)
df_2324 = df_2324.sort_values(by=['기획년도', '시즌', '카테고리', '주차']).reset_index(drop=True)

df_2024_final_cat = set(df_2024_final['카테고리'].unique())

# 결과 확인
print(len(df_2024_final_cat)) #잘 붙었는지 확인

df_2324

91
최종 데이터 shape: (4643, 41)


,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,...,흐림,비,강한 비,눈,강한 눈,진눈깨비,악천후일수,평균기온(도),주차별_목표누적판매율,월별_목표누적판매율
0,2023,2023-08-13,가을_니트 셔츠_라운드_ZB,ZB,01_시즌,가을,니트 셔츠,라운드,1:남성,6310,...,2,0,0,0,0,0,0,28.2,NaN,NaN
1,2023,2023-08-20,가을_니트 셔츠_라운드_ZB,ZB,01_시즌,가을,니트 셔츠,라운드,1:남성,6310,...,3,1,2,0,0,0,2,26.9,NaN,NaN
2,2023,2023-08-27,가을_니트 셔츠_라운드_ZB,ZB,01_시즌,가을,니트 셔츠,라운드,1:남성,6310,...,1,2,1,0,0,0,1,24.2,NaN,NaN
3,2023,2023-09-03,가을_니트 셔츠_라운드_ZB,ZB,01_시즌,가을,니트 셔츠,라운드,1:남성,6310,...,3,0,0,0,0,0,0,26.7,NaN,NaN
4,2023,2023-09-10,가을_니트 셔츠_라운드_ZB,ZB,01_시즌,가을,니트 셔츠,라운드,1:남성,6310,...,1,1,1,0,0,0,1,24.1,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4638,2024,2024-09-01,여름_팬츠_팬츠(일반)_ZG,ZG,01_시즌,여름,팬츠,팬츠(일반),1:남성,2600,...,2,2,0,0,0,0,0,26.2,70.59,80.0
4639,2024,2024-09-08,여름_팬츠_팬츠(일반)_ZG,ZG,01_시즌,여름,팬츠,팬츠(일반),1:남성,2600,...,2,2,0,0,0,0,0,27.6,73.71,80.0
4640,2024,2024-09-15,여름_팬츠_팬츠(일반)_ZG,ZG,01_시즌,여름,팬츠,팬츠(일반),1:남성,2600,...,1,2,1,0,0,0,1,26.5,76.11,80.0
4641,2024,2024-09-22,여름_팬츠_팬츠(일반)_ZG,ZG,01_시즌,여름,팬츠,팬츠(일반),1:남성,2600,...,1,1,0,0,0,0,0,22.2,78.27,80.0


In [10]:
df_2324.to_csv("model_df(목표판매율추가).csv", index=False)

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('./data/model_df_할인율NaN&음수처리.csv')

df_2023 = df[df['기획년도'] == 2023].reset_index(drop=True)
df_2024 = df[df['기획년도'] == 2024].reset_index(drop=True)

season_target = {'봄': 0.80, '여름': 0.80, '가을': 0.65, '겨울': 0.55, '사계절': 0.80}

result = pd.DataFrame()

for season in df_2024['시즌'].unique():
    season_df_2023 = df_2023[(df_2023['시즌'] == season) & (df_2023['총입고수량'] > 0)].sort_values(by='주차').copy()
    season_pattern = season_df_2023.groupby('주차')['누적판매율'].mean().reset_index()
    season_pattern = season_pattern.sort_values(by='주차')['누적판매율'].values

    # 2024년 주차 확보
    season_weeks_2024 = df_2024[df_2024['시즌'] == season][['주차']].drop_duplicates().sort_values(by='주차').reset_index(drop=True)
    total_weeks_2024 = season_weeks_2024.shape[0]
    goal = season_target[season]

    if len(season_pattern) == 0 or total_weeks_2024 == 0:
        continue

    # 패턴 보간 후 goal 맞추기
    scaled_curve = np.interp(
        np.linspace(0, len(season_pattern) - 1, total_weeks_2024),
        np.arange(len(season_pattern)),
        season_pattern
    )
    scaled_curve = scaled_curve / scaled_curve[-1] * goal

    # log 가중치 적용
    log_curve = np.log(np.linspace(1, total_weeks_2024, total_weeks_2024))
    log_curve = (log_curve - log_curve.min()) / (log_curve.max() - log_curve.min())
    log_adjusted_curve = scaled_curve * log_curve
    tail_curve = log_adjusted_curve[1:]
    tail_curve = tail_curve / tail_curve[-1] * goal
    final_curve = np.concatenate([[scaled_curve[0]], tail_curve])

    # 결과 저장
    season_weeks_2024['시즌'] = season
    season_weeks_2024['주차별_목표누적판매율'] = np.round(final_curve * 100, 2)  # 백분율 변환
    season_weeks_2024['월'] = pd.to_datetime(season_weeks_2024['주차'], errors='coerce').dt.month
    result = pd.concat([result, season_weeks_2024], ignore_index=True)

# 월별 목표 계산 (마지막 주차 목표 복사)
result['월별_목표누적판매율'] = np.nan
for season in result['시즌'].unique():
    for month in result[result['시즌'] == season]['월'].dropna().unique():
        month_data = result[(result['시즌'] == season) & (result['월'] == month)].sort_values(by='주차')
        last_week_goal = month_data.iloc[-1]['주차별_목표누적판매율']
        result.loc[(result['시즌'] == season) & (result['월'] == month), '월별_목표누적판매율'] = last_week_goal

# 각 카테고리별 첫 번째 주차 목표 누적판매율을 0으로 변경
result_idx = result.groupby(['시즌'])['주차'].idxmin()
result.loc[result_idx, '주차별_목표누적판매율'] = 0

# 최종 정리
result = result[['월','주차', '시즌', '주차별_목표누적판매율','월별_목표누적판매율']].sort_values(by=['시즌', '주차']).reset_index(drop=True)

# CSV 저장
result.to_csv('시즌별_목표누적판매율.csv', index=False, encoding='utf-8-sig')
# print("저장 완료: 시즌별_목표누적판매율.csv")